# Venusian PCW Detector 科学图表

用原始 1 Hz MAG 和事件清单生成图表。太阳黑子观测、弓激波位置清单与模型参数随仓库提供。全任务连续搜索耗时较长，通过参数单独开启。每组图保存为 PNG 和 PDF，输出在 `build/figures/`。

In [ ]:
# 【原始数据和事件清单】
MAG_DIR = None  # 本机 1 Hz MAG BAM 目录。None 沿用 VENUS_EXPRESS_RAW_DATA_DIR。
EVENTS_CSV = None  # PCW 事件清单。None 使用仓库附带的事件清单。

# 【弓激波三联图】两种 CSV 入口只选一种。都为 None 时读取随附的散点坐标。
SHOCK_POSITIONS_CSV = None  # 弓激波散点坐标 CSV，字段 X_AVSO_Rv、R_AVSO_Rv、period。
SHOCK_CROSSINGS_CSV = None  # 自备 UTC 穿越时刻 CSV，字段 utc、period。需要 MAG 数据。
SHOCK_REFIT_POSITIONS = False  # True 时用散点重新拟合。UTC 入口会自动拟合。

# 【计算范围与运行环境】
EVENT_DAYS = None  # 事件测量日期。None 处理整份清单，单日示例 ["2006-04-24"]。
WORKERS = 4  # 测量事件和搜索时使用的进程数。
DEVICE = "cpu"  # 连续搜索的设备，可选 cpu、mps 或 cuda。

# 【计算和出图】各步骤按开关执行。
RUN_CATALOGUE = True  # 从 MAG 和事件清单重算事件位置、波能及密度。
RUN_MISSION_SEARCH = False  # 用随附权重检出候选，重算曝光及发生率。
SEARCH_DAYS = None  # 仅用于连续搜索。None 搜索 MAG 目录内的全部观测日。
DRAW_MAG_EXAMPLES = True  # 本机有对应日期的 MAG 数据时绘制观测示例。
SHOW_INLINE = True  # True 时在 Notebook 显示图件，文件均保存到 build/figures/。

# 【字体】家族按优先顺序填写。留空时自动使用附带字体及可用系统字体。
FONT_FAMILIES = []
FONT_PATHS = []  # 可添加本机字体目录。

## 准备环境

In [ ]:
import os
from pathlib import Path

from IPython.display import Image, display
from venus_express.repro.settings import BUILD, FIGURES, display_path, read

os.environ.setdefault("MPLCONFIGDIR", str(BUILD / "matplotlib"))
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
if MAG_DIR is not None:
    os.environ["VENUS_EXPRESS_RAW_DATA_DIR"] = str(Path(MAG_DIR).expanduser().resolve())

from venus_express.repro.raw import observation_days
from venusian_pcw import figures
from venusian_pcw.fonts import plot_families
from venusian_pcw.notebook import day_flags, run_cli

font_stack = plot_families(FONT_FAMILIES, FONT_PATHS)
has_mag = bool(os.environ.get("VENUS_EXPRESS_RAW_DATA_DIR"))
if not has_mag:
    print("设置 MAG_DIR 后可运行事件测量、搜索和原始观测示例。")

generated = []
def draw_figure(name, render, *args):
    with figures.figure_style(font_stack):
        render(*args)
    path = FIGURES / f"{name}.png"
    generated.append(path)
    if SHOW_INLINE:
        display(Image(filename=str(path), width=850))

## 太阳活动与参考弓激波

黑子图读取 SILSO 月均观测。弓激波图的三个面板依次为低活动期穿越位置与曲线、高活动期穿越位置与曲线、两条模型曲线对比。

使用散点坐标 CSV 时，默认按随附模型参数复绘。开启 `SHOCK_REFIT_POSITIONS` 可直接用散点重新拟合。自备 UTC 穿越清单时可从 MAG 提取位置并拟合。

In [ ]:
draw_figure("07_solar_cycle", figures.solar_cycle)
shock_name = "09_shock_geometry"
if SHOCK_CROSSINGS_CSV is not None:
    shock_name += "_from_bam"
elif SHOCK_REFIT_POSITIONS:
    shock_name += "_refit"
draw_figure(shock_name, lambda: figures.shock_models(
    SHOCK_POSITIONS_CSV, SHOCK_CROSSINGS_CSV, refit_positions=SHOCK_REFIT_POSITIONS
))

## 事件空间分布和氢密度

从 BAM 逐事件测量；样本满足拟合要求时生成氢密度图。

In [ ]:
catalogue_dir = None
has_annual_energy = False
if RUN_CATALOGUE and has_mag:
    event_flags = ["--events", Path(EVENTS_CSV).expanduser().resolve()] if EVENTS_CSV else []
    run_cli("catalogue", *event_flags, *day_flags(EVENT_DAYS), "--workers", WORKERS)
    catalogue_dir = BUILD / "raw_catalogue"
    if EVENT_DAYS:
        catalogue_dir /= "diagnostic"
    measured = read(catalogue_dir / "summary.json")
    if measured["energy_available"]:
        draw_figure("12_rx_manual_energy", figures.catalogue_map, catalogue_dir)
    if "density" in measured:
        draw_figure("06_h_density", figures.density, catalogue_dir)
    has_annual_energy = "wave_energy" in measured

## 连续检出、空间曝光与年际变化

开启搜索时，由权重和 BAM 重算候选、逐日曝光及空间发生率。

In [ ]:
has_occurrence = RUN_MISSION_SEARCH and has_mag
if has_occurrence:
    run_cli("search", "mission", *day_flags(SEARCH_DAYS), "--workers", WORKERS, "--device", DEVICE)
    run_cli("replay", "--workers", WORKERS)
    draw_figure("11_rx_cluster_occurrence", figures.cluster_map)
if has_occurrence or has_annual_energy:
    draw_figure("05_annual_response", figures.annual_response,
                catalogue_dir if has_annual_energy else None, has_occurrence)

## 原始 MAG 观测示例

In [ ]:
if DRAW_MAG_EXAMPLES and has_mag:
    dates = set(observation_days())
    if "2014-02-22" in dates:
        draw_figure("04_pcw_event", figures.wave_event)
    if "2008-12-03" in dates:
        draw_figure("10_radial_imf_event", figures.radial_event)
    if {"2011-02-20", "2014-02-22", "2009-10-12"} <= dates:
        draw_figure("13_rx_case_orbits", figures.rx_examples)

print(f"本次生成 {len(generated)} 张图。目录：{display_path(FIGURES)}")